<a href="https://colab.research.google.com/github/f0rceFOS/python-practice-algos/blob/master/636-01/hw_6_vinogradov_a_a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Задание для самостоятельной работы (ДЗ 6)

1. Запустите `DBSCAN` на датасете Wine (с параметрами `eps=1.5, min_samples=5`).
2. Посчитайте `silhouette_score` и `adjusted_rand_score`.
3. Сравните с KMeans. Какой алгоритм лучше справился с поиском истинных сортов? Почему?


In [1]:
import numpy as np
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

wine = load_wine()
X = wine.data
y_true = wine.target

X_scaled = StandardScaler().fit_transform(X)

dbscan_labels = DBSCAN(eps=1.5, min_samples=5).fit_predict(X_scaled)

kmeans_labels = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X_scaled)


def evaluate(name, labels, ignore_noise=False):
    """Печатает число кластеров, silhouette_score и adjusted_rand_score."""
    ari = adjusted_rand_score(y_true, labels)

    mask = (labels != -1) if ignore_noise else np.ones(len(labels), dtype=bool)
    selected_labels = labels[mask]
    n_clusters = len(np.unique(selected_labels))
    n_noise = int(np.sum(labels == -1))

    if 2 <= n_clusters < len(selected_labels):
        silhouette = silhouette_score(X_scaled[mask], selected_labels)
        silhouette_text = f"{silhouette:.4f}"
    else:
        silhouette = None
        silhouette_text = "не определён (недостаточно кластеров)"

    print(f"{name}:")
    print(f"  Кластеров: {n_clusters}; шумовых объектов: {n_noise}")
    print(f"  silhouette_score: {silhouette_text}")
    print(f"  adjusted_rand_score: {ari:.4f}")
    return ari, silhouette


dbscan_ari, dbscan_sil = evaluate("DBSCAN", dbscan_labels, ignore_noise=True)
kmeans_ari, kmeans_sil = evaluate("KMeans", kmeans_labels)

print("\nВывод:")
if kmeans_ari > dbscan_ari:
    print("KMeans лучше восстановил истинные сорта Wine по adjusted_rand_score.")
elif dbscan_ari > kmeans_ari:
    print("DBSCAN лучше восстановил истинные сорта Wine по adjusted_rand_score.")
else:
    print("По adjusted_rand_score результаты алгоритмов одинаковы.")

if dbscan_sil is None:
    print("DBSCAN с eps=1.5 и min_samples=5 не сформировал хотя бы два кластера;")
    print("поэтому silhouette_score для него вычислять нельзя.")
    print("В 13-мерном пространстве после стандартизации точки могут быть")
    print("слишком разреженными для выбранного радиуса eps.")

# Сложность в НАИХУДШЕМ случае (n — объектов, d — признаков,
# k — число кластеров, I — число итераций KMeans, R — число запусков):
# StandardScaler: время O(n*d), дополнительная память O(n*d).
# DBSCAN (реализация sklearn): время до O(n^2*d), память до O(n^2).
# KMeans: время O(R*I*n*k*d), память O(n*d + n*k).
# silhouette_score: время O(n^2*d), память до O(n^2).
# Итого для этого решения: время O(n^2*d + R*I*n*k*d),
# память O(n^2 + n*d + n*k) в худшем случае.


DBSCAN:
  Кластеров: 0; шумовых объектов: 178
  silhouette_score: не определён (недостаточно кластеров)
  adjusted_rand_score: 0.0000
KMeans:
  Кластеров: 3; шумовых объектов: 0
  silhouette_score: 0.2849
  adjusted_rand_score: 0.8975

Вывод:
KMeans лучше восстановил истинные сорта Wine по adjusted_rand_score.
DBSCAN с eps=1.5 и min_samples=5 не сформировал хотя бы два кластера;
поэтому silhouette_score для него вычислять нельзя.
В 13-мерном пространстве после стандартизации точки могут быть
слишком разреженными для выбранного радиуса eps.
